# Laboratorijska vježba 4: Neuronske mreže

# Zadatak 1 - Binarna klasifikacija - Spam filter

### a) Učitavanje skupa podataka i prikaz prvih 3 podatka

In [ ]:
import pandas as pd

# Učitavamo kao .txt jer je tako na sistemu, ali je CSV format
spam_df = pd.read_csv('SpamDetectionData.txt')
print('Prvih 3 podatka:')
print(spam_df.head(3))
print('\nKolone u skupu podataka:', spam_df.columns.tolist())
print('Oblik skupa podataka:', spam_df.shape)

### b) Izdvajanje X i y, uklanjanje HTML tagova `<p>` i `</p>`

In [ ]:
X = spam_df['Message'].copy()
y = spam_df['Label'].copy()

# Uklanjanje HTML tagova <p> i </p>
X = X.str.replace('<p>', '', regex=False)
X = X.str.replace('</p>', '', regex=False)

print('Broj slova u prvoj poruci:', len(X.iloc[0]))
print('Broj slova u drugoj poruci:', len(X.iloc[1]))

### c) Podjela skupa podataka na trening i testni set (10% testni)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print('Veličina trening skupa:', len(X_train))
print('Veličina testnog skupa:', len(X_test))

### d) Tokenizacija teksta

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

train_data_seq = tokenizer.texts_to_sequences(X_train)
test_data_seq = tokenizer.texts_to_sequences(X_test)

# Tri najčešće riječi
word_counts = sorted(tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)
print('Tri najčešće riječi:')
for i in range(3):
    print(f'  {i+1}. {word_counts[i][0]} ({word_counts[i][1]} ponavljanja)')

print('\nPrva rečenica iz trening skupa:')
print(X_train.iloc[0])
print('\nFormirani niz cijelih brojeva za nju:')
print(train_data_seq[0])

### e) Vektorizacija podataka i labela

In [ ]:
import numpy as np

def vectorize_sequences(sequences, dimension=4000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        # Uzimamo samo one tokene koji su unutar dimenzije
        valid_tokens = [idx for idx in sequence if idx < dimension]
        results[i, valid_tokens] = 1.
    return results

def vectorize_labels(labels):
    results = np.zeros(len(labels))
    for i, label in enumerate(labels):
        if label.lower() == 'spam':
            results[i] = 1
    return results

x_train = vectorize_sequences(train_data_seq, dimension=4000)
x_test = vectorize_sequences(test_data_seq, dimension=4000)

y_train_vec = vectorize_labels(y_train)
y_test_vec = vectorize_labels(y_test)

print('Vektorizacija završena.')

### f) Definisanje sekvencijalnog Keras modela

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(8, activation='relu', input_shape=(4000,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

### g) Kompajliranje modela

In [ ]:
model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### h) Istrenirati model na 5 epoha i grafički prikaz

In [ ]:
import matplotlib.pyplot as plt

history = model.fit(
    x_train, y_train_vec,
    epochs=5,
    batch_size=128,
    validation_split=0.3
)

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()
plt.show()

### i) Evaluacija modela nad testnim skupom

In [ ]:
results = model.evaluate(x_test, y_test_vec)
print(f'Test loss: {results[0]:.4f}')
print(f'Test accuracy: {results[1]:.4f}')

### j) Predikcija na proizvoljnoj poruci

In [ ]:
test_message = "Congratulations! You have won a free gift card. Click here to claim."
test_seq = tokenizer.texts_to_sequences([test_message])
test_vec = vectorize_sequences(test_seq, dimension=4000)
prediction = model.predict(test_vec)
print(f'Vjerovatnoća da je poruka spam: {prediction[0][0]:.4f}')
print('Klasifikacija:', 'Spam' if prediction[0][0] > 0.5 else 'Ham')

# Zadatak 2 - Višeklasna klasifikacija - StackOverflow pitanja

### a) Učitavanje 'stackoverflow.csv' i prikaz posljednja tri podatka

In [ ]:
df2 = pd.read_csv('stackoverflow.csv')
print(df2.tail(3))

### b) Izdvajanje X i y, broj jedinstvenih labela i jezici

In [ ]:
X2 = df2['post']
y2 = df2['tags']
print('Jedinstvene labele:', y2.unique())
print('Broj jezika:', len(y2.unique()))

### c) One-hot enkodiranje labela

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

encoder = LabelEncoder()
y2_int = encoder.fit_transform(y2)
y2_one_hot = to_categorical(y2_int)
print('Prvih 5 one-hot labela:')
print(y2_one_hot[:5])

### d) Podjela skupa podataka (10% testni)

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2_one_hot, test_size=0.1, random_state=42
)

### e) Tokenizacija (500 najčešćih riječi) i vektorizacija

In [ ]:
max_words = 500
tokenizer2 = Tokenizer(num_words=max_words)
tokenizer2.fit_on_texts(X2_train)

train2_seq = tokenizer2.texts_to_sequences(X2_train)
test2_seq = tokenizer2.texts_to_sequences(X2_test)

x2_train = vectorize_sequences(train2_seq, dimension=max_words)
x2_test = vectorize_sequences(test2_seq, dimension=max_words)

### f) Definisanje sekvencijalnog Keras modela sa 3 Dense sloja

In [ ]:
model2 = Sequential([
    Dense(32, activation='relu', input_shape=(max_words,)),
    Dense(8, activation='relu'),
    Dense(len(y2.unique()), activation='softmax')
])

### g) Kompajliranje modela (adam, categorical_crossentropy)

In [ ]:
model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model2.summary()

### h) Istrenirati model na 8 epoha i grafički prikaz

In [ ]:
history2 = model2.fit(
    x2_train, y2_train,
    epochs=8,
    batch_size=8,
    validation_split=0.25
)

acc2 = history2.history['accuracy']
val_acc2 = history2.history['val_accuracy']
loss2 = history2.history['loss']
val_loss2 = history2.history['val_loss']
epochs2 = range(1, len(acc2) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs2, loss2, 'bo', label='Training loss')
plt.plot(epochs2, val_loss2, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs2, acc2, 'bo', label='Training acc')
plt.plot(epochs2, val_acc2, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()
plt.show()

### i) Evaluacija modela nad testnim skupom

In [ ]:
results2 = model2.evaluate(x2_test, y2_test)
print(f'Test loss: {results2[0]:.4f}')
print(f'Test accuracy: {results2[1]:.4f}')